In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ml-fundamentals-and-applications-2025-07-08/final_proj_data.csv
/kaggle/input/competitions/ml-fundamentals-and-applications-2025-07-08/final_proj_test.csv
/kaggle/input/competitions/ml-fundamentals-and-applications-2025-07-08/final_proj_sample_submission.csv


In [2]:
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
train_file = "/kaggle/input/competitions/ml-fundamentals-and-applications-2025-07-08/final_proj_data.csv"
train_df = pd.read_csv(train_file)
train_df.shape

(10000, 231)

In [3]:
valid_file = "/kaggle/input/competitions/ml-fundamentals-and-applications-2025-07-08/final_proj_test.csv"
competition_test_df = pd.read_csv(valid_file)
competition_test_df.shape

(2500, 230)

In [4]:
sample_file= "/kaggle/input/competitions/ml-fundamentals-and-applications-2025-07-08/final_proj_sample_submission.csv"
sample_submission = pd.read_csv(sample_file)
sample_submission .head()

,index,y
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 231 entries, Var1 to y
dtypes: float64(191), int64(2), object(38)
memory usage: 17.6+ MB


In [6]:
train_df.head()

,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var222,Var223,Var224,Var225,Var226,Var227,Var228,Var229,Var230,y
0,NaN,NaN,NaN,NaN,NaN,812.0,14.0,NaN,NaN,NaN,...,catzS2D,jySVZNlOJy,NaN,xG3x,Aoh3,ZI9m,ib5G6X1eUxUn6,mj86,NaN,0
1,NaN,NaN,NaN,NaN,NaN,2688.0,7.0,NaN,NaN,NaN,...,i06ocsg,LM8l689qOp,NaN,kG3k,WqMG,RAYp,55YFVY9,mj86,NaN,0
2,NaN,NaN,NaN,NaN,NaN,1015.0,14.0,NaN,NaN,NaN,...,P6pu4Vl,LM8l689qOp,NaN,kG3k,Aoh3,ZI9m,R4y5gQQWY8OodqDV,am7c,NaN,0
3,NaN,NaN,NaN,NaN,NaN,168.0,0.0,NaN,NaN,NaN,...,BNrD3Yd,LM8l689qOp,NaN,NaN,FSa2,RAYp,F2FyR07IdsN7I,NaN,NaN,0
4,NaN,NaN,NaN,NaN,NaN,14.0,0.0,NaN,NaN,NaN,...,3B1QowC,LM8l689qOp,NaN,NaN,WqMG,RAYp,F2FyR07IdsN7I,NaN,NaN,0


In [7]:
train_df.isnull().sum()

Var1       9867
Var2       9734
Var3       9734
Var4       9720
Var5       9759
          ...  
Var227        0
Var228        0
Var229     5561
Var230    10000
y             0
Length: 231, dtype: int64

In [8]:
train_df.duplicated().sum()

np.int64(0)

In [9]:
X = train_df.drop(columns="y")
y = train_df["y"]

data_num = X.select_dtypes(include=['float64', 'int64'])
data_cat = X.select_dtypes(include='object')

# 8. Аналіз target

In [10]:
y.unique()

array([0, 1])

In [11]:
# співвідношення класів
y.value_counts(normalize=True)

y
0    0.8695
1    0.1305
Name: proportion, dtype: float64

**Аналіз numerical**

In [12]:
stats = pd.DataFrame({
    "zero_count": (data_num == 0).sum(),
    "nan_count": data_num.isna().sum(),
    "zero_nan_count": (data_num == 0).sum() + data_num.isna().sum(),
})

# чи 0 є справжнім значенням, чи фактично позначає відсутні дані ?

stats["zero_nan_percent"] = (
    stats["zero_nan_count"] / len(data_num) * 100
)

# кількість порожніх колонок
zero_count = (stats["zero_nan_percent"] > 70).sum()

# загальна кількість колонок
col_total = len(data_num.columns)

print(
    f"Всього колонок: {col_total}, "
    f"порожніх: {zero_count}, "
    f"відсоток: {zero_count / col_total * 100:.2f}%"
)

Всього колонок: 192, порожніх: 157, відсоток: 81.77%


**Аналіз категорій**

In [13]:
data_cat.isnull().sum().head()

Var191    9830
Var192      79
Var193       0
Var194    7467
Var195       0
dtype: int64

In [14]:
data_cat.apply(lambda x: x.unique()[:5]).head()

Var191                                          [nan, r__I]
Var192    [KXMrEyXXnK, 8Knvyx875g, MfKrEyQtC3, Qu0qrQKzJ...
Var193     [g62hiBSaKg, 2Knk1KF, RO12, AERks4l, e6CkoqApVR]
Var194                              [SEuy, nan, lvza, CTUH]
Var195    [taul, CiJDdr4TQ0rGERIS, LfvqpCtLOY, I9xt3GDRh...
dtype: object

# 7. Cardinality категоріальних ознак

In [15]:
category_stats = pd.DataFrame({
    "unique": data_cat.nunique(), # кількість унікальних значень у кожному стовпці
    "missing": data_cat.isna().sum()
})

category_stats["missing_percent"] = (
    category_stats["missing"] /
    len(data_cat) * 100
)

#category_stats.sort_values("unique", ascending=False)

In [16]:
print(
    "Категорій <= 10:",
    (category_stats["unique"] <= 10).sum()
)

print(
    "Категорій 11-50:",
    (
        (category_stats["unique"] > 10) &
        (category_stats["unique"] <= 50)
    ).sum()
)

print(
    "Категорій > 50:",
    (category_stats["unique"] > 50).sum()
)

Категорій <= 10: 18
Категорій 11-50: 7
Категорій > 50: 13


In [17]:
missing_stats = pd.DataFrame({
    "missing_count": X.isna().sum(),
    "missing_percent": X.isna().mean() * 100
})

missing_stats = missing_stats.sort_values(
    "missing_percent",
    ascending=False
)

missing_stats.head(20)

,missing_count,missing_percent
Var15,10000,100.00
Var31,10000,100.00
Var32,10000,100.00
Var20,10000,100.00
Var8,10000,100.00
Var39,10000,100.00
Var48,10000,100.00
Var52,10000,100.00
Var79,10000,100.00
Var55,10000,100.00


In [18]:
#X_drop = X.drop(columns=["Var217", "Var200", "Var214", "Var202"])
#data_num = X_drop.select_dtypes(include=['float64', 'int64'])
#data_cat = X_drop.select_dtypes(include='object')

In [19]:
# data_cat["Var217"]

In [20]:
# замінюємо 0 -> NaN
#data_num = data_num.replace(0, np.nan)

# data_num.isna().sum()

In [21]:
# залишили з X тільки ті стовпці, у яких менше 35% пропущених значень
# 0.35
#data_num = data_num[data_num.columns[data_num.isna().mean().lt(0.3)]]

#data_cat = data_cat[data_cat.columns[data_cat.isna().mean().lt(0.3)]]

# 9. Train / validation split

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, # X_drop,
    y,
    test_size=0.20,
    stratify=y, # зберігає приблизно однаковий баланс класів.
    random_state=42
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

X_train: (8000, 230)
X_valid: (2000, 230)


# 10. Preprocessing

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder

**Transformer для numeric features**

In [24]:
from sklearn.base import BaseEstimator, TransformerMixin

class DropHighMissingColumns(BaseEstimator, TransformerMixin):

    def __init__(self, threshold=0.35):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()

        missing_ratio = X.isna().mean()

        self.columns_to_keep_ = missing_ratio[
            missing_ratio < self.threshold
        ].index.tolist()

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        return X[self.columns_to_keep_]

In [25]:
numeric_pipeline = Pipeline([
    (
        "drop_missing",
        DropHighMissingColumns(threshold=0.7)
    ),    
    (
        "imputer",
        SimpleImputer(strategy="median")
        # KBinsDiscretizer(encode='ordinal').fit(X_train[num_cols])
    ),
    (
        "scaler",
        StandardScaler()
    )
])

**Hybrid transformer для categorical features**

In [26]:
class HybridCategoricalEncoder(
    BaseEstimator,
    TransformerMixin
):

    def __init__(
        self,
        missing_threshold=0.70,
        cardinality_threshold=50,
        random_state=42
    ):
        self.missing_threshold = missing_threshold
        self.cardinality_threshold = cardinality_threshold
        self.random_state = random_state


    def _prepare_columns(self, X):

        X = pd.DataFrame(X).copy()

        missing_ratio = X.isna().mean()

        self.columns_to_keep_ = missing_ratio[
            missing_ratio < self.missing_threshold
        ].index.tolist()

        X = X[self.columns_to_keep_]

        cardinality = X.nunique(
            dropna=True
        )

        self.low_card_cols_ = cardinality[
            cardinality <= self.cardinality_threshold
        ].index.tolist()

        self.high_card_cols_ = cardinality[
            cardinality > self.cardinality_threshold
        ].index.tolist()

        return X

    def _create_onehot_encoder(self):
        return OneHotEncoder(
            handle_unknown="ignore",
            #handle_unknown="infrequent_if_exist", # +++
            #min_frequency=5, # +++            
            #drop="if_binary", # ---
            drop=None,
            sparse_output=False
        )

    def _create_target_encoder(self):
        return TargetEncoder(
            target_type="binary",
            smooth="auto", # +++
            #cv=5, # +++
            shuffle=True, # +++
            random_state=self.random_state
        )
    
    def fit(self, X, y=None):

        if y is None:
            raise ValueError(
                "Target y is required for TargetEncoder."
            )
            
        X = self._prepare_columns(X)

        if self.low_card_cols_:

            self.low_imputer_ = SimpleImputer(
                strategy="most_frequent"
            )

            X_low = self.low_imputer_.fit_transform(
                X[self.low_card_cols_]
            )

            self.onehot_ = self._create_onehot_encoder()
            self.onehot_.fit(X_low)


        if self.high_card_cols_:

            self.high_imputer_ = SimpleImputer(
                strategy="most_frequent"
            )

            X_high = self.high_imputer_.fit_transform(
                X[self.high_card_cols_]
            )

            self.target_encoder_ = self._create_target_encoder()

            self.target_encoder_.fit(
                X_high,
                y
            )

        return self


    def transform(self, X):

        X = pd.DataFrame(X).copy()

        X = X[self.columns_to_keep_]

        parts = []

        if self.low_card_cols_:

            X_low = self.low_imputer_.transform(
                X[self.low_card_cols_]
            )

            X_low = self.onehot_.transform(
                X_low
            )

            parts.append(X_low)


        if self.high_card_cols_:

            X_high = self.high_imputer_.transform(
                X[self.high_card_cols_]
            )

            X_high = self.target_encoder_.transform(
                X_high
            )

            parts.append(X_high)


        return np.hstack(parts)


    def fit_transform(
        self,
        X,
        y=None,
        **fit_params
    ):

        if y is None:
            raise ValueError(
                "Target y is required."
            )

        X = self._prepare_columns(X)

        parts = []

        if self.low_card_cols_:

            self.low_imputer_ = SimpleImputer(
                strategy="most_frequent"
            )

            X_low = self.low_imputer_.fit_transform(
                X[self.low_card_cols_]
            )

            self.onehot_ = self._create_onehot_encoder()

            X_low = self.onehot_.fit_transform(
                X_low
            )

            parts.append(X_low)


        if self.high_card_cols_:

            self.high_imputer_ = SimpleImputer(
                strategy="most_frequent"
            )

            X_high = self.high_imputer_.fit_transform(
                X[self.high_card_cols_]
            )

            self.target_encoder_ = self._create_target_encoder()

            X_high = self.target_encoder_.fit_transform(
                X_high,
                y
            )

            parts.append(X_high)


        return np.hstack(parts)

In [27]:
# DEL

categorical_pipeline = Pipeline([
    (
        "drop_missing",
        DropHighMissingColumns(threshold=0.7)
    ),    
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        # враховуючи що цільова змінна має 0 та 1 задаємо параметри
        TargetEncoder(target_type="binary",random_state=42)
        #WOEEncoder()
    )
])

In [28]:
categorical_pipeline = Pipeline([
    (
        "hybrid_encoder",
        HybridCategoricalEncoder(
            missing_threshold=0.70,
            cardinality_threshold=50,
            random_state=42
        )
    )
])

In [29]:
preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        data_num.columns
    ),
    (
        "cat",
        categorical_pipeline,
        data_cat.columns
    )  
])

# 11. Baseline — Logistic Regression

In [30]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression( #KNN, SVN            
            C=3, # +++
            penalty="l1", # +++
            solver='liblinear', # +++
            class_weight="balanced",
            max_iter=3000, # ---
            random_state=42

        )
    )
])

In [31]:
logistic_model.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('drop_missing',
                                                                   DropHighMissingColumns(threshold=0.7)),
                                                                  ('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Var1', 'Var2', 'Var3', 'Var4', 'Var5', 'Var6', 'Var7', 'Var8', 'Var9',
       'Var10',
       ...
       'Var183', 'Var184', 'Var185', 'Var186', 'Var187', 'Var188...
       'Var205', 'Var206', 'Var207', 'Var208', 'Var210', 'Var211', 'Var212',
       'Var213', 'Var214', 'Var215', 'Var216', 'Var217', 'Var218', 'Var219',
       'Var220', 'Var221', 'Var222', 'Var223', 'Var224', 'Var225', 'Var226',
       'Var227', 'Var228', 'Var229'],
      dtype='object'))])),
                ('classifier',
                 LogisticRegression(C=3, class_weight='balanced', max_iter=3000,
                                    penalty='l1', random_state=42,
                                    solver='liblinear'))])

In [32]:
y_pred = logistic_model.predict(X_valid)

In [33]:
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

In [34]:
baseline_score = balanced_accuracy_score(y_valid,y_pred)

print(f"Balanced Accuracy: {baseline_score:.4f}")

Balanced Accuracy: 0.8126


In [35]:
print(classification_report(y_valid,y_pred))

              precision    recall  f1-score   support

           0       0.97      0.76      0.86      1739
           1       0.35      0.86      0.50       261

    accuracy                           0.78      2000
   macro avg       0.66      0.81      0.68      2000
weighted avg       0.89      0.78      0.81      2000



In [36]:
cm = confusion_matrix(y_valid,y_pred)

print('Confusion Matrix:\n',cm)

Confusion Matrix:
 [[1327  412]
 [  36  225]]


**Підбір параметрів:**

Експеримент із порогом видалення ознак за часткою пропущених значень показав, що найкращий результат отримано при порозі 70%: середня Balanced Accuracy становила 0.7977 ± 0.0118. Більш агресивне видалення ознак при порогах 20–50% погіршувало результат, що свідчить про наявність корисної інформації навіть в ознаках зі значною кількістю пропусків. Водночас збереження майже всіх ознак при порозі 100% знизило Balanced Accuracy до 0.7804, що може бути пов'язано з додаванням шумових або малоінформативних ознак. Тому для подальших експериментів обрано поріг 70%.

Оптимізація параметрів регуляризації Logistic Regression дозволила підвищити середню Balanced Accuracy з 0.7780 до **0.7855**. Найкращою комбінацією виявилися C=3 та L1-регуляризація. Подальше збільшення C до 10 не покращило результат, хоча training score залишався високим, що може свідчити про початок перенавчання. L1-регуляризація також потенційно виконує додатковий відбір ознак через обнулення частини коефіцієнтів. Тому модель C=3, penalty='l1' обрана як новий основний кандидат для подальшої оптимізації ваг класів та порогу класифікації.

Оптимізація порогу класифікації показала максимальну Balanced Accuracy 0.7865 при threshold=**0.43**, порівняно з 0.7855 при стандартному порозі 0.50. Зниження порогу збільшило recall міноритарного класу 1 з 0.8414 до 0.8897, одночасно зменшивши recall класу 0 з 0.7296 до 0.6834. Оскільки приріст Balanced Accuracy становить лише близько 0.001, оптимізація threshold не дала суттєвого покращення загальної якості моделі. Водночас вона продемонструвала можливість керувати компромісом між чутливістю до двох класів.

Порівняння різних значень cardinality_threshold показало, що найкращі результати отримано при cardinality_threshold = 50. Цей варіант забезпечив найвищий середній показник Balanced Accuracy за результатами 5-fold крос-валідації (0.8041) та найкращий результат на валідаційній вибірці (0.8126). Зменшення порогу до 20 або 10 призвело до незначного погіршення якості моделі, що свідчить про втрату корисної інформації під час кодування категоріальних ознак. Використання One-Hot Encoding для ознак із кількістю категорій до 50 та Target Encoding для ознак із більшою кардинальністю виявилося найбільш ефективним підходом. Отже, для подальших експериментів і порівняння моделей доцільно використовувати саме Hybrid Encoding із cardinality_threshold = 50 як базову конфігурацію.

# 12. Cross-validation function

In [37]:
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

In [38]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [39]:
results = []

def evaluate_model(name, model, X, y, cv):
    
    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="balanced_accuracy",
        n_jobs=-1
    )
    
    result = {
        "model": name,
        "mean_balanced_accuracy": scores.mean(),
        "std": scores.std()
    }
    
    results.append(result)
    
    print(name)
    print("Scores:", scores)
    print(f"Mean: {scores.mean():.4f}")
    print(f"STD: {scores.std():.4f}")

# 13. Cross-validation

In [87]:
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

svd_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "svd",
        TruncatedSVD(
            n_components=200,
            random_state=42
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=3,
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        )
    )
])

In [89]:
from sklearn.decomposition import PCA

pca_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    ("scale_all", StandardScaler()),
    (
        "pca",
        PCA(
            n_components=0.95,
            random_state=42
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=3,
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        )
    )
])

In [91]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

linear_svc_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LinearSVC(
            C=0.3,
            class_weight={0: 1, 1: 8},
            loss="squared_hinge",
            dual="auto",
            max_iter=10000,
            random_state=42
        )
    )
])

In [86]:
evaluate_model(
    "Logistic Regression",
    logistic_model,
    X,
    y,
    cv
)

Logistic Regression
Scores: [0.81966339 0.79629373 0.78403055 0.82847631 0.79159644]
Mean: 0.8040
STD: 0.0171


In [88]:
evaluate_model(
    "SVD + Logistic Regression",
    svd_model,
    X,
    y,
    cv
)

SVD + Logistic Regression
Scores: [0.81956645 0.79131222 0.78316798 0.8270387  0.79188396]
Mean: 0.8026
STD: 0.0173


In [90]:
evaluate_model(
    "PCA + Logistic Regression",
    pca_model,
    X,
    y,
    cv
)

PCA + Logistic Regression
Scores: [0.796481   0.80213449 0.78786196 0.82234472 0.79360578]
Mean: 0.8005
STD: 0.0119


In [92]:
evaluate_model(
    "linearSVC",
    linear_svc_model,
    X,
    y,
    cv
)

linearSVC
Scores: [0.80806228 0.7948462  0.79321802 0.82166282 0.79685555]
Mean: 0.8029
STD: 0.0107


# 18. Таблиця результатів

In [93]:
# 18. Таблиця результатів
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "mean_balanced_accuracy",
    ascending=False
)

results_df

,model,mean_balanced_accuracy,std
0,Logistic Regression,0.804012,0.017067
6,Logistic Regression,0.804012,0.017067
9,linearSVC,0.802929,0.010713
5,linearSVC,0.802929,0.010713
1,SVD + Logistic Regression,0.802594,0.017349
7,SVD + Logistic Regression,0.802594,0.017349
3,linearSVC,0.801327,0.015783
4,linearSVC,0.801327,0.015783
2,PCA + Logistic Regression,0.800486,0.011860
8,PCA + Logistic Regression,0.800486,0.011860


In [73]:
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

best_svc_model = Pipeline([
    ("preprocessor", preprocessor),

    ("model", LinearSVC(
        C=0.3,
        class_weight={0: 1, 1: 8},
        loss="squared_hinge",
        dual="auto",
        max_iter=10000,
        random_state=42
    ))
])

best_svc_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('drop_missing',
                                                                   DropHighMissingColumns(threshold=0.7)),
                                                                  ('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Var1', 'Var2', 'Var3', 'Var4', 'Var5', 'Var6', 'Var7', 'Var8', 'Var9',
       'Var10',
       ...
       'Var183', 'Var184', 'Var185', 'Var186', 'Var187', 'Var188...
       'Var198', 'Var199', 'Var200', 'Var201', 'Var202', 'Var203', 'Var204',
       'Var205', 'Var206', 'Var207', 'Var208', 'Var210', 'Var211', 'Var212',
       'Var213', 'Var214', 'Var215', 'Var216', 'Var217', 'Var218', 'Var219',
       'Var220', 'Var221', 'Var222', 'Var223', 'Var224', 'Var225', 'Var226',
       'Var227', 'Var228', 'Var229'],
      dtype='object'))])),
                ('model',
                 LinearSVC(C=0.3, class_weight={0: 1, 1: 8}, max_iter=10000,
                           random_state=42))])

In [74]:
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

svc_pred = best_svc_model.predict(X_valid)

svc_ba = balanced_accuracy_score(
    y_valid,
    svc_pred
)

print("LinearSVC validation BA:", svc_ba)
print(confusion_matrix(y_valid, svc_pred))
print(classification_report(y_valid, svc_pred))

LinearSVC validation BA: 0.8234848935509244
[[1265  474]
 [  21  240]]
              precision    recall  f1-score   support

           0       0.98      0.73      0.84      1739
           1       0.34      0.92      0.49       261

    accuracy                           0.75      2000
   macro avg       0.66      0.82      0.66      2000
weighted avg       0.90      0.75      0.79      2000



In [75]:
import numpy as np
import pandas as pd

svc_scores = best_svc_model.decision_function(X_valid)

threshold_results = []

for threshold in np.arange(-1.0, 1.01, 0.02):
    pred = (svc_scores >= threshold).astype(int)

    ba = balanced_accuracy_score(
        y_valid,
        pred
    )

    threshold_results.append({
        "threshold": threshold,
        "balanced_accuracy": ba
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results.sort_values(
    "balanced_accuracy",
    ascending=False
).head(10)

,threshold,balanced_accuracy
50,8.881784e-16,0.823485
47,-6.000000e-02,0.820988
49,-2.000000e-02,0.819460
46,-8.000000e-02,0.818303
48,-4.000000e-02,0.817925
51,2.000000e-02,0.817357
56,1.200000e-01,0.815070
45,-1.000000e-01,0.814565
52,4.000000e-02,0.814485
53,6.000000e-02,0.814391


In [76]:
best_threshold_row = threshold_results.loc[
    threshold_results["balanced_accuracy"].idxmax()
]

best_svc_threshold = best_threshold_row["threshold"]
best_svc_threshold_ba = best_threshold_row["balanced_accuracy"]

print("Best threshold:", best_svc_threshold)
print("Best BA:", best_svc_threshold_ba)

Best threshold: 8.881784197001252e-16
Best BA: 0.8234848935509244


In [78]:
svc_pred_tuned = (
    svc_scores >= best_svc_threshold
).astype(int)

In [80]:
logistic_pred = logistic_model.predict(X_valid)

logistic_ba = balanced_accuracy_score(
    y_valid,
    logistic_pred
)

svc_default_ba = balanced_accuracy_score(
    y_valid,
    svc_pred
)

svc_tuned_ba = balanced_accuracy_score(
    y_valid,
    svc_pred_tuned
)

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "LinearSVC, threshold=0",
        "LinearSVC, tuned threshold"
    ],
    "Balanced Accuracy": [
        logistic_ba,
        svc_default_ba,
        svc_tuned_ba
    ]
})

comparison.sort_values(
    "Balanced Accuracy",
    ascending=False
)

ValueError: X has 256 features, but LogisticRegression is expecting 262 features as input.

# 12. Fit на всьому train

In [48]:
final_model = logistic_model

final_model.fit(X, y)

test_predictions = final_model.predict(
    competition_test_df
).astype(int)

# 13. Submission

In [49]:
submission = sample_submission.copy()
submission["y"] = test_predictions

#submission.to_csv("/kaggle/working/submission.csv", index=False)

In [50]:
print(submission.head())
print(submission.shape)
print(submission["y"].value_counts())
#print(submission.isna().sum())

   index  y
0      0  0
1      1  0
2      2  0
3      3  0
4      4  1
(2500, 2)
y
0    1733
1     767
Name: count, dtype: int64


In [51]:
assert len(submission) == len(competition_test_df)
assert submission["y"].isin([0, 1]).all()
assert submission.isna().sum().sum() == 0

# CatBoost

In [52]:
X_cb = X.copy()

cat_features = X_cb.select_dtypes(exclude="number").columns.tolist()

for col in cat_features:
    X_cb[col] = (
        X_cb[col]
        .fillna("MISSING")
        .astype(str)
    )

X_valid_cb = X_valid.copy()

for col in cat_features:
    X_valid_cb[col] = (
        X_valid_cb[col]
        .fillna("MISSING")
        .astype(str)
    )

In [53]:
# Створюємо копії, щоб не змінювати початкові дані
X_train_cb = X_train.copy()
X_valid_cb = X_valid.copy()

# Назви категоріальних колонок
cat_features = X_train_cb.select_dtypes(exclude="number").columns.tolist()

# CatBoost не приймає NaN у категоріальних колонках,
# тому замінюємо їх текстовим значенням
for col in cat_features:
    X_train_cb[col] = (
        X_train_cb[col]
        .fillna("MISSING")
        .astype(str)
    )

    X_valid_cb[col] = (
        X_valid_cb[col]
        .fillna("MISSING")
        .astype(str)
    )

In [54]:
from catboost import CatBoostClassifier
from sklearn.metrics import balanced_accuracy_score

final_model = CatBoostClassifier(
    iterations=305,
    learning_rate=0.05,
    depth=6,
    class_weights=[1, 8],
    l2_leaf_reg=3,
    loss_function="Logloss",
    eval_metric="BalancedAccuracy",
    random_seed=42,
    allow_writing_files=False,
    verbose=100
)

final_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_features
    #cat_features=cat_features,
    #eval_set=(X_valid_cb, y_valid),
    #early_stopping_rounds=100,
    #use_best_model=True
)

pred = final_model.predict(X_valid_cb)

ba = balanced_accuracy_score(y_valid,pred)

print(f"Balanced Accuracy = {ba:.4f}")

0:	learn: 0.7281406	total: 123ms	remaining: 37.3s
100:	learn: 0.8880880	total: 5.34s	remaining: 10.8s
200:	learn: 0.9149220	total: 10.7s	remaining: 5.53s
300:	learn: 0.9376095	total: 16.2s	remaining: 215ms
304:	learn: 0.9382564	total: 16.4s	remaining: 0us
Balanced Accuracy = 0.9019


Для побудови моделі класифікації було проведено послідовний підбір основних гіперпараметрів CatBoostClassifier з використанням крос-валідації та метрики Balanced Accuracy, оскільки цільова змінна має незбалансований розподіл класів.

На першому етапі було досліджено вплив параметра class_weights. Найкращий результат показало співвідношення [1,8], яке забезпечило середнє значення Balanced Accuracy = 0.8960. Це підтвердило, що додаткове збільшення ваги міноритарного класу дозволяє покращити його розпізнавання без суттєвого погіршення загальної якості моделі.

Далі було виконано підбір глибини дерев (depth). Порівняння значень від 4 до 8 показало, що найкращу якість забезпечує depth = 6. Менші значення не дозволяли моделі повністю описати залежності між ознаками, тоді як більші значення призводили до початку перенавчання та не покращували результат.

Наступним кроком було дослідження параметра learning_rate. Було встановлено, що значення 0.05 забезпечує найкращий баланс між швидкістю навчання та здатністю моделі до узагальнення. Менше значення (0.01) призводило до недостатнього навчання моделі, а більше (0.10) не давало додаткового приросту якості.

Також було проаналізовано параметр l2_leaf_reg, який виконує роль L2-регуляризації. Найкращий результат було отримано при l2_leaf_reg = 3, тоді як сильніша регуляризація дещо знижувала якість моделі.

Після підбору гіперпараметрів модель була перевірена на відкладеній валідаційній вибірці з використанням механізму Early Stopping. Початково було задано 1000 ітерацій, однак навчання автоматично завершилося після відсутності покращення протягом 100 ітерацій. Найкращий результат було досягнуто на 304-й ітерації, після чого CatBoost автоматично скоротив модель до 305 дерев. Це свідчить про те, що подальше збільшення кількості дерев лише покращувало якість на навчальній вибірці, але вже не покращувало узагальнюючу здатність моделі.

У результаті було отримано:

середнє значення Balanced Accuracy за крос-валідацією — 0.8980;
Standard Deviation = 0.0106, що свідчить про стабільність моделі на різних розбиттях даних;
Balanced Accuracy = 0.9019 на незалежній валідаційній вибірці.

Різниця між результатом крос-валідації (0.8980) та валідаційної вибірки (0.9019) становить лише 0.0039, що є меншою за стандартне відхилення крос-валідації. Це підтверджує відсутність суттєвого перенавчання та хорошу здатність моделі узагальнювати нові дані.

In [55]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_valid, pred))

[[1531  208]
 [  20  241]]


In [56]:
from sklearn.metrics import classification_report

print(classification_report(y_valid, pred))

              precision    recall  f1-score   support

           0       0.99      0.88      0.93      1739
           1       0.54      0.92      0.68       261

    accuracy                           0.89      2000
   macro avg       0.76      0.90      0.80      2000
weighted avg       0.93      0.89      0.90      2000



In [57]:
# competition_test_df — це final_proj_test.csv
X_test_cb = competition_test_df.copy()

# Ті самі ознаки й той самий порядок, що у train
X_test_cb = X_test_cb.reindex(columns=X_train_cb.columns)

# Та сама обробка категоріальних ознак
for col in cat_features:
    X_test_cb[col] = (
        X_test_cb[col]
        .fillna("MISSING")
        .astype(str)
    )

# Прогноз для Kaggle test
test_pred = final_model.predict(X_test_cb)
test_pred = np.asarray(test_pred).reshape(-1).astype(int)

print("Кількість прогнозів:", len(test_pred))
print("Кількість рядків submission:", len(sample_submission))

assert len(test_pred) == len(sample_submission)

Кількість прогнозів: 2500
Кількість рядків submission: 2500


In [58]:
# Формування submission
submission = sample_submission.copy()
# submission["y"] = pred
submission["y"] = test_pred

print(submission.shape)
print(submission.head())
print(submission["y"].value_counts())

#submission.to_csv("submission.csv",index=False)
#print("Submission saved.")

(2500, 2)
   index  y
0      0  0
1      1  0
2      2  0
3      3  0
4      4  0
y
0    1952
1     548
Name: count, dtype: int64
